In [ ]:
import numpy as np
import pandas as pd

In [ ]:
def probablity(x, data):
    data=np.array(data)
    if len(data) == 0:
        return 0
    return np.sum(data==x)/len(data)

In [ ]:
import numpy as np

def entropy(y):
    if len(y) == 0:
        return 0

    classes = np.unique(y)
    ent = 0
    
    for c in classes:
        p = probablity(c, y)
        if p > 0:
            ent += p * np.log2(p)   # ✅ better to use log2
    
    return -ent


def gini_impurity(y):
    if len(y) == 0:
        return 0

    classes = np.unique(y)
    gi = 0

    for c in classes:
        gi += probablity(c, y) ** 2

    return 1 - gi


def information_gain(x, y, feature, impurity='gini'):
    
    if impurity == 'gini':
        parent_impurity = gini_impurity(y)
        impurity_func = gini_impurity

    elif impurity == 'entropy':
        parent_impurity = entropy(y)
        impurity_func = entropy

    else:
        raise ValueError('Invalid impurity!')

    values = np.unique(x[feature])

    weighted_impurity = 0

    for v in values:
        mask = (x[feature] == v)
        subset_y = y[mask]

        weight = len(subset_y) / len(y)
        weighted_impurity += weight * impurity_func(subset_y)

    return parent_impurity - weighted_impurity

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

n_samples = 20000

# Categories
age_group = np.random.choice(['young', 'adult', 'senior'], n_samples)
income_level = np.random.choice(['low', 'medium', 'high'], n_samples)
device_type = np.random.choice(['mobile', 'desktop', 'tablet'], n_samples)
region = np.random.choice(['urban', 'semi-urban', 'rural'], n_samples)
browsing_time = np.random.choice(['low', 'medium', 'high'], n_samples)
ad_clicked = np.random.choice(['yes', 'no'], n_samples)

# Target logic
purchase = []

for i in range(n_samples):
    score = 0
    
    if income_level[i] == 'high':
        score += 2
    elif income_level[i] == 'medium':
        score += 1

    if browsing_time[i] == 'high':
        score += 2
    elif browsing_time[i] == 'medium':
        score += 1

    if ad_clicked[i] == 'yes':
        score += 2

    if region[i] == 'urban':
        score += 1

    if device_type[i] == 'desktop':
        score += 1

    # Decision threshold
    purchase.append(1 if score >= 4 else 0)



# Create DataFrame
df = pd.DataFrame({
    'age_group': age_group,
    'income_level': income_level,
    'device_type': device_type,
    'region': region,
    'browsing_time': browsing_time,
    'ad_clicked': ad_clicked,
    'purchase': purchase
})

print(df.head())
print(df['purchase'].value_counts())

In [ ]:
df

In [ ]:
def build_tree(x, y, features, depth, impurity_function='gini'):
    if len(set(y)) == 1:
        return y.iloc[0]
    
    if depth == 0 or len(features) == 0:
        y.mode()[0]
    
    igs = [information_gain(x, y, feature, impurity_function) for feature in features]
    best_feature = features[np.argmax(igs)]

    tree = {best_feature: {}}

    values = np.unique(x[best_feature])

    remaining_features = features.drop(best_feature)

    for v in values:
        subset = x[x[best_feature] == v]
        sub_y = y[subset.index]

        if len(subset) == 0:
            tree[best_feature][v] = y.mode()[0]
        else:
            subtree = build_tree(
                subset.drop(columns=[best_feature]),
                sub_y,
                remaining_features,
                depth - 1,
                impurity_function
            )
            tree[best_feature][v] = subtree

    return tree

In [ ]:
def predict(tree, sample):
    if not isinstance(tree, dict):
        return tree

    feature = next(iter(tree))
    value = sample[feature]

    if value in tree[feature]:
        return predict(tree[feature][value], sample)
    else:
        return 0  # fallback

In [ ]:
def accuracy(pred, y):
    return np.sum(pred.to_numpy() == y) / len(y)

In [ ]:
def decision_tree_classifier(dataset, unseen_data, builder='gini', height=5):
    dataset = dataset.sample(frac=1, random_state=42).reset_index(drop=True)

    features = dataset.columns[:-1]

    split = int(0.7 * len(dataset))

    train_data = dataset.iloc[:split]
    test_data = dataset.iloc[split:]
    
    x_train = train_data.iloc[:, :-1]
    y_train = train_data.iloc[:, -1]

    x_test = test_data.iloc[:, :-1]
    y_test = test_data.iloc[:, -1]

    if builder not in ['gini', 'entropy']:
        raise ValueError('Invalid builder function chosen!')

    decision_tree = build_tree(
        x_train,
        y_train,
        features,
        height,
        builder
    )
    
    test_predictions = x_test.apply(lambda row: predict(decision_tree, row), axis=1)

    print(f'Accuracy: {accuracy(test_predictions, y_test)}')

    if isinstance(unseen_data, pd.Series):
        actual_predictions = predict(decision_tree, unseen_data)
    else:
        actual_predictions = unseen_data.apply(lambda row: predict(decision_tree, row), axis=1)

    return actual_predictions

In [ ]:
import pandas as pd

# Manually crafted unseen dataset
unseen_df = pd.DataFrame([
    # Strong positive cases (should be 1)
    ['young', 'high', 'desktop', 'urban', 'high', 'yes'],   # score = 2+2+2+1+1 = 8 → 1
    ['adult', 'medium', 'desktop', 'urban', 'high', 'yes'], # score = 1+2+2+1+1 = 7 → 1
    ['senior', 'high', 'mobile', 'urban', 'medium', 'yes'], # score = 2+1+2+1 = 6 → 1

    # Borderline but positive
    ['adult', 'medium', 'desktop', 'semi-urban', 'high', 'yes'], # score = 1+2+2+1 = 6 → 1

    # Negative cases (should be 0)
    ['young', 'low', 'mobile', 'rural', 'low', 'no'],       # score = 0 → 0
    ['senior', 'low', 'tablet', 'rural', 'medium', 'no'],   # score = 1 → 0
    ['adult', 'medium', 'mobile', 'rural', 'low', 'no'],    # score = 1 → 0

    # Borderline negative
    ['young', 'medium', 'mobile', 'semi-urban', 'medium', 'no'], # score = 2 → 0

], columns=[
    'age_group',
    'income_level',
    'device_type',
    'region',
    'browsing_time',
    'ad_clicked'
])

# Expected outputs based on logic
expected = [1, 1, 1, 1, 0, 0, 0, 0]

print(unseen_df)
print("Expected:", expected)

In [ ]:
predictions = decision_tree_classifier(df, unseen_df, builder='gini', height=5)

print("Model Predictions:", list(predictions))
print("Expected:", expected)